In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Main Calorify path
BASE_DIR = Path("/content/drive/MyDrive/Calorify")

# Phase 2 folders
PHASE2_DIR = BASE_DIR / "phase2_text_calorie"
DATA_DIR = PHASE2_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
CLEAN_DIR = DATA_DIR / "cleaned"
RESULTS_DIR = PHASE2_DIR / "results"
MODELS_DIR = PHASE2_DIR / "models"
LOGS_DIR = PHASE2_DIR / "logs"

# Create folders
folders = [
    PHASE2_DIR,
    DATA_DIR,
    RAW_DIR,
    CLEAN_DIR,
    RESULTS_DIR,
    MODELS_DIR,
    LOGS_DIR
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("✅ Phase 2 folders created successfully:")
for folder in folders:
    print(folder)

Mounted at /content/drive
✅ Phase 2 folders created successfully:
/content/drive/MyDrive/Calorify/phase2_text_calorie
/content/drive/MyDrive/Calorify/phase2_text_calorie/data
/content/drive/MyDrive/Calorify/phase2_text_calorie/data/raw
/content/drive/MyDrive/Calorify/phase2_text_calorie/data/cleaned
/content/drive/MyDrive/Calorify/phase2_text_calorie/results
/content/drive/MyDrive/Calorify/phase2_text_calorie/models
/content/drive/MyDrive/Calorify/phase2_text_calorie/logs


In [3]:
# Step 1.2 — Define food classes and search queries

FOOD_CLASSES = [
    "burger",
    "pizza",
    "fries",
    "pasta",
    "rice",
    "chicken",
    "salad",
    "sandwich",
    "sushi",
    "steak"
]

SEARCH_QUERIES = {
    "burger": [
        "burger recipe",
        "cheeseburger recipe",
        "double burger recipe",
        "beef burger recipe",
        "chicken burger recipe"
    ],
    "pizza": [
        "pizza recipe",
        "pepperoni pizza recipe",
        "margherita pizza recipe",
        "cheese pizza recipe",
        "vegetable pizza recipe"
    ],
    "fries": [
        "fries recipe",
        "french fries recipe",
        "loaded fries recipe",
        "cheese fries recipe",
        "potato fries recipe"
    ],
    "pasta": [
        "pasta recipe",
        "chicken pasta recipe",
        "alfredo pasta recipe",
        "tomato pasta recipe",
        "spaghetti recipe"
    ],
    "rice": [
        "rice recipe",
        "fried rice recipe",
        "chicken rice recipe",
        "vegetable rice recipe",
        "rice bowl recipe"
    ],
    "chicken": [
        "chicken recipe",
        "grilled chicken recipe",
        "fried chicken recipe",
        "chicken breast recipe",
        "roasted chicken recipe"
    ],
    "salad": [
        "salad recipe",
        "caesar salad recipe",
        "chicken salad recipe",
        "green salad recipe",
        "greek salad recipe"
    ],
    "sandwich": [
        "sandwich recipe",
        "club sandwich recipe",
        "chicken sandwich recipe",
        "cheese sandwich recipe",
        "grilled sandwich recipe"
    ],
    "sushi": [
        "sushi recipe",
        "salmon sushi recipe",
        "tuna sushi recipe",
        "sushi roll recipe",
        "california roll recipe"
    ],
    "steak": [
        "steak recipe",
        "grilled steak recipe",
        "beef steak recipe",
        "sirloin steak recipe",
        "steak with sauce recipe"
    ]
}

TARGET_RECORDS_PER_CLASS = 50

print(" Food classes and search queries defined.")
print("Number of classes:", len(FOOD_CLASSES))
print("Target records per class:", TARGET_RECORDS_PER_CLASS)

for food_class in FOOD_CLASSES:
    print(f"{food_class}: {len(SEARCH_QUERIES[food_class])} queries")

 Food classes and search queries defined.
Number of classes: 10
Target records per class: 50
burger: 5 queries
pizza: 5 queries
fries: 5 queries
pasta: 5 queries
rice: 5 queries
chicken: 5 queries
salad: 5 queries
sandwich: 5 queries
sushi: 5 queries
steak: 5 queries


In [5]:
# Step 1.3 — Spoonacular API credentials setup

import getpass

SPOONACULAR_API_KEY = getpass.getpass("Enter your Spoonacular API KEY: ")

BASE_URL = "https://api.apilayer.com/spoonacular"

print(" Spoonacular credentials loaded.")
print("Base URL:", BASE_URL)
print("API KEY length:", len(SPOONACULAR_API_KEY))

Enter your Spoonacular API KEY: ··········
 Spoonacular credentials loaded.
Base URL: https://api.apilayer.com/spoonacular
API KEY length: 32


In [6]:
# Step 1.4 — Test Spoonacular API connection

import requests
import json

test_url = f"{BASE_URL}/recipes/complexSearch"

headers = {
    "apikey": SPOONACULAR_API_KEY
}

params = {
    "query": "burger",
    "number": 2,
    "addRecipeInformation": True,
    "addRecipeNutrition": True
}

response = requests.get(test_url, headers=headers, params=params)

print("Status code:", response.status_code)

try:
    data = response.json()
    print("Response keys:", data.keys())

    if "results" in data:
        print("Number of results:", len(data["results"]))

        if len(data["results"]) > 0:
            first_recipe = data["results"][0]
            print("\nFirst recipe sample:")
            print("ID:", first_recipe.get("id"))
            print("Title:", first_recipe.get("title"))
            print("Image:", first_recipe.get("image"))

            nutrition = first_recipe.get("nutrition", {})
            nutrients = nutrition.get("nutrients", [])

            print("\nFirst 5 nutrients:")
            for nutrient in nutrients[:5]:
                print(nutrient.get("name"), nutrient.get("amount"), nutrient.get("unit"))
    else:
        print("No 'results' key found.")
        print(json.dumps(data, indent=2)[:1500])

except Exception as e:
    print("Could not parse JSON.")
    print("Error:", e)
    print(response.text[:1500])

Status code: 200
Response keys: dict_keys(['results', 'offset', 'number', 'totalResults'])
Number of results: 2

First recipe sample:
ID: 642540
Title: Falafel Burgers
Image: https://img.spoonacular.com/recipes/642540-312x231.jpg

First 5 nutrients:
Calories 709.5 kcal
Fat 35.46 g
Saturated Fat 4.99 g
Carbohydrates 80.23 g
Net Carbohydrates 65.83 g


In [7]:
# Step 1.5 — Extract one recipe into a structured record

def get_nutrient_amount(nutrients, nutrient_name):
    """
    Extract nutrient amount by name from Spoonacular nutrients list.
    """
    for nutrient in nutrients:
        if nutrient.get("name", "").lower() == nutrient_name.lower():
            return nutrient.get("amount")
    return None


def extract_recipe_record(recipe, food_class, query):
    """
    Convert one Spoonacular recipe JSON object into a structured dataset record.
    """
    nutrition = recipe.get("nutrition", {})
    nutrients = nutrition.get("nutrients", [])

    extended_ingredients = recipe.get("extendedIngredients", [])

    ingredients_list = []
    for item in extended_ingredients:
        ingredient_text = item.get("original") or item.get("name")
        if ingredient_text:
            ingredients_list.append(ingredient_text)

    ingredients_text = " ".join(ingredients_list)

    record = {
        "source_api": "Spoonacular_API_Layer",
        "food_class": food_class,
        "query": query,
        "recipe_id": recipe.get("id"),
        "recipe_name": recipe.get("title"),
        "ingredients_text": ingredients_text,
        "calories": get_nutrient_amount(nutrients, "Calories"),
        "protein": get_nutrient_amount(nutrients, "Protein"),
        "fat": get_nutrient_amount(nutrients, "Fat"),
        "carbs": get_nutrient_amount(nutrients, "Carbohydrates"),
        "image_url": recipe.get("image"),
        "source_url": recipe.get("sourceUrl"),
    }

    return record


# Test extraction on the first recipe from previous API response
first_recipe = data["results"][0]
sample_record = extract_recipe_record(
    recipe=first_recipe,
    food_class="burger",
    query="burger"
)

print(" Sample structured record:")
for key, value in sample_record.items():
    print(f"{key}: {value}")

 Sample structured record:
source_api: Spoonacular_API_Layer
food_class: burger
query: burger
recipe_id: 642540
recipe_name: Falafel Burgers
ingredients_text: 
calories: 709.5
protein: 23.31
fat: 35.46
carbs: 80.23
image_url: https://img.spoonacular.com/recipes/642540-312x231.jpg
source_url: https://www.foodista.com/recipe/FBWMX8MY/falafel-burgers


In [8]:
# Step 1.6 — Test API request with ingredients included

test_url = f"{BASE_URL}/recipes/complexSearch"

headers = {
    "apikey": SPOONACULAR_API_KEY
}

params = {
    "query": "burger",
    "number": 2,
    "addRecipeInformation": True,
    "addRecipeNutrition": True,
    "fillIngredients": True
}

response = requests.get(test_url, headers=headers, params=params)

print("Status code:", response.status_code)

data_with_ingredients = response.json()

print("Response keys:", data_with_ingredients.keys())
print("Number of results:", len(data_with_ingredients.get("results", [])))

if len(data_with_ingredients.get("results", [])) > 0:
    first_recipe = data_with_ingredients["results"][0]

    print("\nFirst recipe keys:")
    print(first_recipe.keys())

    print("\nTitle:", first_recipe.get("title"))

    print("\nextendedIngredients exists:", "extendedIngredients" in first_recipe)
    print("Number of extendedIngredients:", len(first_recipe.get("extendedIngredients", [])))

    print("\nFirst 5 ingredients:")
    for ing in first_recipe.get("extendedIngredients", [])[:5]:
        print("-", ing.get("original") or ing.get("name"))

Status code: 200
Response keys: dict_keys(['results', 'offset', 'number', 'totalResults'])
Number of results: 2

First recipe keys:
dict_keys(['id', 'image', 'imageType', 'title', 'readyInMinutes', 'servings', 'sourceUrl', 'vegetarian', 'vegan', 'glutenFree', 'dairyFree', 'veryHealthy', 'cheap', 'veryPopular', 'sustainable', 'lowFodmap', 'weightWatcherSmartPoints', 'gaps', 'preparationMinutes', 'cookingMinutes', 'aggregateLikes', 'healthScore', 'creditsText', 'license', 'sourceName', 'pricePerServing', 'extendedIngredients', 'nutrition', 'summary', 'cuisines', 'dishTypes', 'diets', 'occasions', 'language', 'spoonacularScore', 'spoonacularSourceUrl', 'usedIngredientCount', 'missedIngredientCount', 'missedIngredients', 'likes', 'usedIngredients', 'unusedIngredients'])

Title: Falafel Burgers

extendedIngredients exists: True
Number of extendedIngredients: 15

First 5 ingredients:
- 2 cans garbanzo beans (chickpeas), drained and rinsed
- 1 tablespoon chili powder
- 1 tablespoon coriander


In [9]:
# Step 1.7 — Final recipe extraction function

def get_nutrient_amount(nutrients, nutrient_name):
    """
    Extract nutrient amount by nutrient name from Spoonacular nutrients list.
    Example: Calories, Protein, Fat, Carbohydrates
    """
    for nutrient in nutrients:
        if nutrient.get("name", "").lower() == nutrient_name.lower():
            return nutrient.get("amount")
    return None


def extract_recipe_record(recipe, food_class, query):
    """
    Convert one Spoonacular recipe JSON object into one structured dataset record.
    """
    nutrition = recipe.get("nutrition", {})
    nutrients = nutrition.get("nutrients", [])

    extended_ingredients = recipe.get("extendedIngredients", [])

    ingredients_list = []
    for item in extended_ingredients:
        ingredient_text = item.get("original") or item.get("name")
        if ingredient_text:
            ingredients_list.append(ingredient_text)

    ingredients_text = " ".join(ingredients_list)

    record = {
        "source_api": "Spoonacular_API_Layer",
        "food_class": food_class,
        "query": query,
        "recipe_id": recipe.get("id"),
        "recipe_name": recipe.get("title"),
        "ingredients_text": ingredients_text,
        "calories": get_nutrient_amount(nutrients, "Calories"),
        "protein": get_nutrient_amount(nutrients, "Protein"),
        "fat": get_nutrient_amount(nutrients, "Fat"),
        "carbs": get_nutrient_amount(nutrients, "Carbohydrates"),
        "servings": recipe.get("servings"),
        "ready_in_minutes": recipe.get("readyInMinutes"),
        "image_url": recipe.get("image"),
        "source_url": recipe.get("sourceUrl") or recipe.get("spoonacularSourceUrl"),
    }

    return record


# Test final extraction
first_recipe = data_with_ingredients["results"][0]

sample_record = extract_recipe_record(
    recipe=first_recipe,
    food_class="burger",
    query="burger"
)

print("Final structured recipe record:")
for key, value in sample_record.items():
    print(f"{key}: {value}")

print("\nIngredients length:", len(sample_record["ingredients_text"]))
print("Calories:", sample_record["calories"])

Final structured recipe record:
source_api: Spoonacular_API_Layer
food_class: burger
query: burger
recipe_id: 642540
recipe_name: Falafel Burgers
ingredients_text: 2 cans garbanzo beans (chickpeas), drained and rinsed 1 tablespoon chili powder 1 tablespoon coriander 1 tablespoon cumin 4 tablespoons flour 1 large handful parsley, chopped 2 cloves garlic, grated or finely chopped 2 Zest and juice of lemons 4 pita pockets 1 small red onion, chopped Salt and pepper, to taste 1/2 cup tahini 1 1/2 teaspoons turmeric 1/4 cup vegetable oil 3 tablespoons water
calories: 709.5
protein: 23.31
fat: 35.46
carbs: 80.23
servings: 4
ready_in_minutes: 45
image_url: https://img.spoonacular.com/recipes/642540-312x231.jpg
source_url: https://www.foodista.com/recipe/FBWMX8MY/falafel-burgers

Ingredients length: 393
Calories: 709.5


In [11]:
# Step 1.8 — Collect multiple recipes for one class

import time
import pandas as pd

def collect_recipes_for_class(food_class, queries, target_records=10, number_per_query=5):
    """
    Collect recipe records for one food class using multiple search queries.
    This is a test collector for one class before collecting all classes.
    """
    records = []
    seen_recipe_ids = set()

    search_url = f"{BASE_URL}/recipes/complexSearch"

    headers = {
        "apikey": SPOONACULAR_API_KEY
    }

    for query in queries:
        print(f"\n Searching query: {query}")

        params = {
            "query": query,
            "number": number_per_query,
            "addRecipeInformation": True,
            "addRecipeNutrition": True,
            "fillIngredients": True
        }

        try:
            response = requests.get(search_url, headers=headers, params=params)
            print("Status code:", response.status_code)

            if response.status_code != 200:
                print("Request failed:")
                print(response.text[:500])
                continue

            data = response.json()
            recipes = data.get("results", [])

            print("Recipes returned:", len(recipes))

            for recipe in recipes:
                recipe_id = recipe.get("id")

                if recipe_id in seen_recipe_ids:
                    continue

                record = extract_recipe_record(
                    recipe=recipe,
                    food_class=food_class,
                    query=query
                )

                # Basic validity check
                if (
                    record["recipe_name"]
                    and record["ingredients_text"]
                    and record["calories"] is not None
                ):
                    records.append(record)
                    seen_recipe_ids.add(recipe_id)

                if len(records) >= target_records:
                    break

            print(f"Collected so far for {food_class}: {len(records)}")

            if len(records) >= target_records:
                break

            time.sleep(1)

        except Exception as e:
            print(" Error while collecting:", e)

    return records


# Test collection for burger only
burger_records = collect_recipes_for_class(
    food_class="burger",
    queries=SEARCH_QUERIES["burger"],
    target_records=10,
    number_per_query=5
)

burger_df = pd.DataFrame(burger_records)

print("\n Burger collection finished.")
print("Shape:", burger_df.shape)

display(burger_df.head())


 Searching query: burger recipe
Status code: 200
Recipes returned: 5
Collected so far for burger: 5

 Searching query: cheeseburger recipe
Status code: 200
Recipes returned: 3
Collected so far for burger: 8

 Searching query: double burger recipe
Status code: 200
Recipes returned: 0
Collected so far for burger: 8

 Searching query: beef burger recipe
Status code: 200
Recipes returned: 5
Collected so far for burger: 10

 Burger collection finished.
Shape: (10, 14)


,source_api,food_class,query,recipe_id,recipe_name,ingredients_text,calories,protein,fat,carbs,servings,ready_in_minutes,image_url,source_url
0,Spoonacular_API_Layer,burger,burger recipe,642540,Falafel Burgers,"2 cans garbanzo beans (chickpeas), drained and...",709.50,23.31,35.46,80.23,4,45,https://img.spoonacular.com/recipes/642540-312...,https://www.foodista.com/recipe/FBWMX8MY/falaf...
1,Spoonacular_API_Layer,burger,burger recipe,775621,Gorgonzola & Grilled Peach Burger,about 2 cups of arugula 4 burger buns Extra Vi...,970.65,50.62,69.59,34.02,4,45,https://img.spoonacular.com/recipes/775621-312...,http://mixandmatchmama.blogspot.com/2016/06/go...
2,Spoonacular_API_Layer,burger,burger recipe,649141,La Bella Italian Turkey Burger,a pinch of black pepper 4 burger buns 1 egg 1 ...,845.75,63.94,51.40,33.43,4,30,https://img.spoonacular.com/recipes/649141-312...,https://www.foodista.com/recipe/PLJM7MZK/la-be...
3,Spoonacular_API_Layer,burger,burger recipe,651190,Masala-Tofu Burger,1 medium white onion- finely chopped shopping ...,494.93,18.73,10.53,84.97,4,45,https://img.spoonacular.com/recipes/651190-312...,https://www.foodista.com/recipe/DGH6HX5S/masal...
4,Spoonacular_API_Layer,burger,burger recipe,650377,Low Carb Brunch Burger,1 avocado sliced 4 strips bacon freshly ground...,1024.58,53.80,82.60,17.58,2,30,https://img.spoonacular.com/recipes/650377-312...,https://www.foodista.com/recipe/5SPTY657/low-c...


In [12]:
# Step 1.9 — Pilot collection for all classes

all_pilot_records = []

for food_class in FOOD_CLASSES:
    print("\n====================================")
    print("Collecting pilot records for class:", food_class)
    print("====================================")

    class_records = collect_recipes_for_class(
        food_class=food_class,
        queries=SEARCH_QUERIES[food_class],
        target_records=10,
        number_per_query=5
    )

    all_pilot_records.extend(class_records)

    print("Finished class:", food_class)
    print("Records collected:", len(class_records))

pilot_df = pd.DataFrame(all_pilot_records)

print("\nPilot collection finished.")
print("Total shape:", pilot_df.shape)

print("\nRecords per class:")
print(pilot_df["food_class"].value_counts())

pilot_output_path = RAW_DIR / "text_calorie_pilot_raw.csv"
pilot_df.to_csv(pilot_output_path, index=False)

print("\nPilot dataset saved to:")
print(pilot_output_path)

display(pilot_df.head(10))



 Searching query: burger recipe
Status code: 200
Recipes returned: 5
Collected so far for burger: 5

 Searching query: cheeseburger recipe
Status code: 200
Recipes returned: 3
Collected so far for burger: 8

 Searching query: double burger recipe
Status code: 200
Recipes returned: 0
Collected so far for burger: 8

 Searching query: beef burger recipe
Status code: 200
Recipes returned: 5
Collected so far for burger: 10
Finished class: burger
Records collected: 10


 Searching query: pizza recipe
Status code: 200
Recipes returned: 5
Collected so far for pizza: 5

 Searching query: pepperoni pizza recipe
Status code: 200
Recipes returned: 5
Collected so far for pizza: 10
Finished class: pizza
Records collected: 10


 Searching query: fries recipe
Status code: 200
Recipes returned: 5
Collected so far for fries: 5

 Searching query: french fries recipe
Status code: 200
Recipes returned: 5
Collected so far for fries: 7

 Searching query: loaded fries recipe
Status code: 200
Recipes returne

,source_api,food_class,query,recipe_id,recipe_name,ingredients_text,calories,protein,fat,carbs,servings,ready_in_minutes,image_url,source_url
0,Spoonacular_API_Layer,burger,burger recipe,642540,Falafel Burgers,"2 cans garbanzo beans (chickpeas), drained and...",709.50,23.31,35.46,80.23,4,45,https://img.spoonacular.com/recipes/642540-312...,https://www.foodista.com/recipe/FBWMX8MY/falaf...
1,Spoonacular_API_Layer,burger,burger recipe,775621,Gorgonzola & Grilled Peach Burger,about 2 cups of arugula 4 burger buns Extra Vi...,970.65,50.62,69.59,34.02,4,45,https://img.spoonacular.com/recipes/775621-312...,http://mixandmatchmama.blogspot.com/2016/06/go...
2,Spoonacular_API_Layer,burger,burger recipe,649141,La Bella Italian Turkey Burger,a pinch of black pepper 4 burger buns 1 egg 1 ...,845.75,63.94,51.40,33.43,4,30,https://img.spoonacular.com/recipes/649141-312...,https://www.foodista.com/recipe/PLJM7MZK/la-be...
3,Spoonacular_API_Layer,burger,burger recipe,651190,Masala-Tofu Burger,1 medium white onion- finely chopped shopping ...,494.93,18.73,10.53,84.97,4,45,https://img.spoonacular.com/recipes/651190-312...,https://www.foodista.com/recipe/DGH6HX5S/masal...
4,Spoonacular_API_Layer,burger,burger recipe,650377,Low Carb Brunch Burger,1 avocado sliced 4 strips bacon freshly ground...,1024.58,53.80,82.60,17.58,2,30,https://img.spoonacular.com/recipes/650377-312...,https://www.foodista.com/recipe/5SPTY657/low-c...
5,Spoonacular_API_Layer,burger,cheeseburger recipe,635350,Blue Cheese Burgers,3/4 tsp kosher salt 1 pound(s) ground sirloin ...,658.94,44.76,36.95,33.80,4,45,https://img.spoonacular.com/recipes/635350-312...,https://www.foodista.com/recipe/ZVXPQHB5/blue-...
6,Spoonacular_API_Layer,burger,cheeseburger recipe,681713,Everything Bagel Cheese Burger,1 pound ground beef 2 Tablespoons Everything B...,820.48,42.16,37.17,75.63,4,45,https://img.spoonacular.com/recipes/681713-312...,https://fullbellysisters.blogspot.com/2015/08/...
7,Spoonacular_API_Layer,burger,cheeseburger recipe,1487865,Easy Cheeseburger Casserole,1 can cream of mushroom soup 1 cup uncooked lo...,436.52,25.20,23.41,29.89,6,100,https://img.spoonacular.com/recipes/1487865-31...,https://www.pinkwhen.com/easy-cheeseburger-cas...
8,Spoonacular_API_Layer,burger,beef burger recipe,632342,An American Beef Burger,550gr / 1.2 lb good quality beef mince 1 x egg...,823.59,39.49,39.04,76.67,4,45,https://img.spoonacular.com/recipes/632342-312...,https://www.foodista.com/recipe/XVQW332P/an-am...
9,Spoonacular_API_Layer,burger,beef burger recipe,663050,Tex-Mex Burger,"1 avocado, thinly sliced 1 tsp Chili Powder ¾ ...",884.46,50.53,61.30,32.26,4,15,https://img.spoonacular.com/recipes/663050-312...,https://www.foodista.com/recipe/NSYCCHLT/tex-m...


In [13]:
# Step 1.10 — Improve sushi search queries

SEARCH_QUERIES["sushi"] = [
    "sushi recipe",
    "sushi roll recipe",
    "california roll recipe",
    "salmon sushi recipe",
    "tuna sushi recipe",
    "maki roll recipe",
    "nigiri recipe",
    "spicy tuna roll recipe",
    "vegetable sushi recipe",
    "homemade sushi recipe",
    "shrimp sushi recipe",
    "avocado sushi roll recipe"
]

print("Updated sushi queries:")
for query in SEARCH_QUERIES["sushi"]:
    print("-", query)

print("Number of sushi queries:", len(SEARCH_QUERIES["sushi"]))

Updated sushi queries:
- sushi recipe
- sushi roll recipe
- california roll recipe
- salmon sushi recipe
- tuna sushi recipe
- maki roll recipe
- nigiri recipe
- spicy tuna roll recipe
- vegetable sushi recipe
- homemade sushi recipe
- shrimp sushi recipe
- avocado sushi roll recipe
Number of sushi queries: 12


In [14]:
# Test improved sushi collection

sushi_records_test = collect_recipes_for_class(
    food_class="sushi",
    queries=SEARCH_QUERIES["sushi"],
    target_records=10,
    number_per_query=5
)

sushi_test_df = pd.DataFrame(sushi_records_test)

print("Sushi test collection finished.")
print("Shape:", sushi_test_df.shape)

if not sushi_test_df.empty:
    print("\nRecipe names:")
    for name in sushi_test_df["recipe_name"].tolist():
        print("-", name)

display(sushi_test_df.head(10))


 Searching query: sushi recipe
Status code: 200
Recipes returned: 5
Collected so far for sushi: 5

 Searching query: sushi roll recipe
Status code: 200
Recipes returned: 4
Collected so far for sushi: 5

 Searching query: california roll recipe
Status code: 200
Recipes returned: 0
Collected so far for sushi: 5

 Searching query: salmon sushi recipe
Status code: 200
Recipes returned: 1
Collected so far for sushi: 5

 Searching query: tuna sushi recipe
Status code: 200
Recipes returned: 3
Collected so far for sushi: 6

 Searching query: maki roll recipe
Status code: 200
Recipes returned: 2
Collected so far for sushi: 7

 Searching query: nigiri recipe
Status code: 200
Recipes returned: 0
Collected so far for sushi: 7

 Searching query: spicy tuna roll recipe
Status code: 200
Recipes returned: 1
Collected so far for sushi: 7

 Searching query: vegetable sushi recipe
Status code: 200
Recipes returned: 3
Collected so far for sushi: 7

 Searching query: homemade sushi recipe
Status code: 200

,source_api,food_class,query,recipe_id,recipe_name,ingredients_text,calories,protein,fat,carbs,servings,ready_in_minutes,image_url,source_url
0,Spoonacular_API_Layer,sushi,sushi recipe,648506,Japanese Sushi,Cooked octopus Cooked prawns Raw tuna Salmon S...,570.85,69.89,12.73,38.42,1,45,https://img.spoonacular.com/recipes/648506-312...,https://www.foodista.com/recipe/ZHC2WBHW/japan...
1,Spoonacular_API_Layer,sushi,sushi recipe,650651,Make It Quick Italian Shrimp Rolls,1 cup low sugar spaghetti sauce 8 ounces shrim...,380.88,37.24,8.37,40.21,2,45,https://img.spoonacular.com/recipes/650651-312...,https://www.foodista.com/recipe/476BQSYP/make-...
2,Spoonacular_API_Layer,sushi,sushi recipe,654563,Panko Crusted Shrimp Rolls,"1 tablespoon barbecue sauce 1 celery stalk, mi...",660.98,38.69,19.50,81.57,4,45,https://img.spoonacular.com/recipes/654563-312...,https://www.foodista.com/recipe/LLLRLRBR/panko...
3,Spoonacular_API_Layer,sushi,sushi recipe,658295,Rice-less Spicy Tuna Hand Rolls,•1/2 Persian Cucumber sliced into thin strips....,144.50,28.97,1.96,0.88,6,45,https://img.spoonacular.com/recipes/658295-312...,https://www.foodista.com/recipe/55G2QSXT/rice-...
4,Spoonacular_API_Layer,sushi,sushi recipe,648742,Kappa Maki,"2 Japanese cucumber, cut into long sticks 4 in...",351.69,6.82,0.63,77.24,8,45,https://img.spoonacular.com/recipes/648742-312...,https://www.foodista.com/recipe/VJ3JTZWL/kappa...
5,Spoonacular_API_Layer,sushi,tuna sushi recipe,1697537,Hawaiian Poke (Aloha Poke),"1 lb sushi grade tuna, cubed 2 Tbsp soy sauce ...",75.96,11.50,2.44,1.58,8,60,https://img.spoonacular.com/recipes/1697537-31...,https://maplewoodroad.com/what-is-hawaiian-pok...
6,Spoonacular_API_Layer,sushi,maki roll recipe,634165,Banana Prawn Rolls,2Gently mix in banana chunks. 5In a heated non...,342.28,24.36,4.86,52.84,3,45,https://img.spoonacular.com/recipes/634165-312...,https://www.foodista.com/recipe/WWGGTJ6V/banan...


In [15]:
# Step 1.11 — Collector with pagination support

def collect_recipes_for_class_paginated(
    food_class,
    queries,
    target_records=50,
    number_per_request=10,
    max_offsets_per_query=5,
    sleep_seconds=1
):
    """
    Collect recipe records for one food class using multiple search queries and pagination.
    This version uses offset to retrieve more results from each query.
    """
    records = []
    seen_recipe_ids = set()

    search_url = f"{BASE_URL}/recipes/complexSearch"

    headers = {
        "apikey": SPOONACULAR_API_KEY
    }

    for query in queries:
        print("\nSearching query:", query)

        for offset_step in range(max_offsets_per_query):
            offset = offset_step * number_per_request

            params = {
                "query": query,
                "number": number_per_request,
                "offset": offset,
                "addRecipeInformation": True,
                "addRecipeNutrition": True,
                "fillIngredients": True
            }

            try:
                response = requests.get(search_url, headers=headers, params=params)
                print("Status code:", response.status_code, "| Offset:", offset)

                if response.status_code != 200:
                    print("Request failed:")
                    print(response.text[:500])
                    continue

                data = response.json()
                recipes = data.get("results", [])

                print("Recipes returned:", len(recipes))

                if len(recipes) == 0:
                    break

                for recipe in recipes:
                    recipe_id = recipe.get("id")

                    if recipe_id in seen_recipe_ids:
                        continue

                    record = extract_recipe_record(
                        recipe=recipe,
                        food_class=food_class,
                        query=query
                    )

                    if (
                        record["recipe_name"]
                        and record["ingredients_text"]
                        and record["calories"] is not None
                    ):
                        records.append(record)
                        seen_recipe_ids.add(recipe_id)

                    if len(records) >= target_records:
                        break

                print("Collected so far for", food_class + ":", len(records))

                if len(records) >= target_records:
                    break

                time.sleep(sleep_seconds)

            except Exception as e:
                print("Error while collecting:", e)

        if len(records) >= target_records:
            break

    return records

In [16]:
# Test paginated sushi collection

sushi_records_paginated = collect_recipes_for_class_paginated(
    food_class="sushi",
    queries=SEARCH_QUERIES["sushi"],
    target_records=20,
    number_per_request=10,
    max_offsets_per_query=5,
    sleep_seconds=1
)

sushi_paginated_df = pd.DataFrame(sushi_records_paginated)

print("Sushi paginated collection finished.")
print("Shape:", sushi_paginated_df.shape)

if not sushi_paginated_df.empty:
    print("\nRecipe names:")
    for name in sushi_paginated_df["recipe_name"].tolist():
        print("-", name)

display(sushi_paginated_df.head(20))


Searching query: sushi recipe
Status code: 200 | Offset: 0
Recipes returned: 6
Collected so far for sushi: 6
Status code: 200 | Offset: 10
Recipes returned: 0

Searching query: sushi roll recipe
Status code: 200 | Offset: 0
Recipes returned: 4
Collected so far for sushi: 6
Status code: 200 | Offset: 10
Recipes returned: 0

Searching query: california roll recipe
Status code: 200 | Offset: 0
Recipes returned: 0

Searching query: salmon sushi recipe
Status code: 200 | Offset: 0
Recipes returned: 1
Collected so far for sushi: 6
Status code: 200 | Offset: 10
Recipes returned: 0

Searching query: tuna sushi recipe
Status code: 200 | Offset: 0
Recipes returned: 3
Collected so far for sushi: 6
Status code: 200 | Offset: 10
Recipes returned: 0

Searching query: maki roll recipe
Status code: 200 | Offset: 0
Recipes returned: 2
Collected so far for sushi: 7
Status code: 200 | Offset: 10
Recipes returned: 0

Searching query: nigiri recipe
Status code: 200 | Offset: 0
Recipes returned: 0

Searchi

,source_api,food_class,query,recipe_id,recipe_name,ingredients_text,calories,protein,fat,carbs,servings,ready_in_minutes,image_url,source_url
0,Spoonacular_API_Layer,sushi,sushi recipe,648506,Japanese Sushi,Cooked octopus Cooked prawns Raw tuna Salmon S...,570.85,69.89,12.73,38.42,1,45,https://img.spoonacular.com/recipes/648506-312...,https://www.foodista.com/recipe/ZHC2WBHW/japan...
1,Spoonacular_API_Layer,sushi,sushi recipe,650651,Make It Quick Italian Shrimp Rolls,1 cup low sugar spaghetti sauce 8 ounces shrim...,380.88,37.24,8.37,40.21,2,45,https://img.spoonacular.com/recipes/650651-312...,https://www.foodista.com/recipe/476BQSYP/make-...
2,Spoonacular_API_Layer,sushi,sushi recipe,654563,Panko Crusted Shrimp Rolls,"1 tablespoon barbecue sauce 1 celery stalk, mi...",660.98,38.69,19.50,81.57,4,45,https://img.spoonacular.com/recipes/654563-312...,https://www.foodista.com/recipe/LLLRLRBR/panko...
3,Spoonacular_API_Layer,sushi,sushi recipe,658295,Rice-less Spicy Tuna Hand Rolls,•1/2 Persian Cucumber sliced into thin strips....,144.50,28.97,1.96,0.88,6,45,https://img.spoonacular.com/recipes/658295-312...,https://www.foodista.com/recipe/55G2QSXT/rice-...
4,Spoonacular_API_Layer,sushi,sushi recipe,648742,Kappa Maki,"2 Japanese cucumber, cut into long sticks 4 in...",351.69,6.82,0.63,77.24,8,45,https://img.spoonacular.com/recipes/648742-312...,https://www.foodista.com/recipe/VJ3JTZWL/kappa...
5,Spoonacular_API_Layer,sushi,sushi recipe,1697537,Hawaiian Poke (Aloha Poke),"1 lb sushi grade tuna, cubed 2 Tbsp soy sauce ...",75.96,11.50,2.44,1.58,8,60,https://img.spoonacular.com/recipes/1697537-31...,https://maplewoodroad.com/what-is-hawaiian-pok...
6,Spoonacular_API_Layer,sushi,maki roll recipe,634165,Banana Prawn Rolls,2Gently mix in banana chunks. 5In a heated non...,342.28,24.36,4.86,52.84,3,45,https://img.spoonacular.com/recipes/634165-312...,https://www.foodista.com/recipe/WWGGTJ6V/banan...


In [17]:
# Step 1.12 — Broaden sushi queries and inspect unique returned recipes

SUSHI_BROAD_QUERIES = [
    "sushi",
    "sushi recipe",
    "sushi rice",
    "sushi roll",
    "maki",
    "maki roll",
    "kappa maki",
    "temaki",
    "hand roll",
    "spicy tuna roll",
    "tuna roll",
    "salmon roll",
    "california roll",
    "avocado roll",
    "cucumber roll",
    "shrimp roll",
    "nori roll",
    "seaweed roll",
    "japanese roll",
    "homemade sushi"
]

def inspect_sushi_query_results(queries, number_per_request=10):
    """
    Inspect Spoonacular results for broader sushi-related queries.
    This does not save a dataset. It only checks unique recipe names and IDs.
    """
    search_url = f"{BASE_URL}/recipes/complexSearch"
    headers = {"apikey": SPOONACULAR_API_KEY}

    seen_ids = set()
    inspection_rows = []

    for query in queries:
        params = {
            "query": query,
            "number": number_per_request,
            "addRecipeInformation": True,
            "addRecipeNutrition": True,
            "fillIngredients": True
        }

        response = requests.get(search_url, headers=headers, params=params)
        print("\nQuery:", query)
        print("Status code:", response.status_code)

        if response.status_code != 200:
            print("Request failed:")
            print(response.text[:500])
            continue

        data = response.json()
        recipes = data.get("results", [])
        print("Recipes returned:", len(recipes))

        new_count = 0

        for recipe in recipes:
            recipe_id = recipe.get("id")
            title = recipe.get("title", "")

            if recipe_id in seen_ids:
                continue

            seen_ids.add(recipe_id)
            new_count += 1

            inspection_rows.append({
                "query": query,
                "recipe_id": recipe_id,
                "recipe_name": title,
                "calories": get_nutrient_amount(
                    recipe.get("nutrition", {}).get("nutrients", []),
                    "Calories"
                ),
                "source_url": recipe.get("sourceUrl") or recipe.get("spoonacularSourceUrl")
            })

        print("New unique recipes from this query:", new_count)
        print("Total unique recipes so far:", len(seen_ids))

        time.sleep(1)

    inspection_df = pd.DataFrame(inspection_rows)
    return inspection_df


sushi_inspection_df = inspect_sushi_query_results(
    queries=SUSHI_BROAD_QUERIES,
    number_per_request=10
)

print("\nSushi inspection finished.")
print("Shape:", sushi_inspection_df.shape)

print("\nRecipe names:")
for name in sushi_inspection_df["recipe_name"].tolist():
    print("-", name)

display(sushi_inspection_df)


Query: sushi
Status code: 200
Recipes returned: 6
New unique recipes from this query: 6
Total unique recipes so far: 6

Query: sushi recipe
Status code: 200
Recipes returned: 6
New unique recipes from this query: 0
Total unique recipes so far: 6

Query: sushi rice
Status code: 200
Recipes returned: 3
New unique recipes from this query: 0
Total unique recipes so far: 6

Query: sushi roll
Status code: 200
Recipes returned: 4
New unique recipes from this query: 0
Total unique recipes so far: 6

Query: maki
Status code: 200
Recipes returned: 10
New unique recipes from this query: 9
Total unique recipes so far: 15

Query: maki roll
Status code: 200
Recipes returned: 2
New unique recipes from this query: 1
Total unique recipes so far: 16

Query: kappa maki
Status code: 200
Recipes returned: 1
New unique recipes from this query: 0
Total unique recipes so far: 16

Query: temaki
Status code: 200
Recipes returned: 0
New unique recipes from this query: 0
Total unique recipes so far: 16

Query: h

,query,recipe_id,recipe_name,calories,source_url
0,sushi,648506,Japanese Sushi,570.85,https://www.foodista.com/recipe/ZHC2WBHW/japan...
1,sushi,650651,Make It Quick Italian Shrimp Rolls,380.88,https://www.foodista.com/recipe/476BQSYP/make-...
2,sushi,654563,Panko Crusted Shrimp Rolls,660.98,https://www.foodista.com/recipe/LLLRLRBR/panko...
3,sushi,658295,Rice-less Spicy Tuna Hand Rolls,144.50,https://www.foodista.com/recipe/55G2QSXT/rice-...
4,sushi,648742,Kappa Maki,351.69,https://www.foodista.com/recipe/VJ3JTZWL/kappa...
5,sushi,1697537,Hawaiian Poke (Aloha Poke),75.96,https://maplewoodroad.com/what-is-hawaiian-pok...
6,maki,642129,Easy To Make Spring Rolls,161.61,https://www.foodista.com/recipe/B5HHJWNP/easy-...
7,maki,1046982,How to Make the Perfect Sweet Potato Sloppy Joes,679.31,https://www.pinkwhen.com/how-to-make-the-perfe...
8,maki,715449,How to Make OREO Turkeys for Thanksgiving,835.44,https://www.pinkwhen.com/oreo-cookie-balls-tha...
9,maki,1050445,How to Make the Best Crock Pot Roast,254.55,https://www.pinkwhen.com/crock-pot-roast/
